In [0]:
from pyspark.sql.functions import rand
from pyspark.sql import functions as F

In [0]:
%python
spark.sql("CREATE CATALOG IF NOT EXISTS gs_carbon")
spark.sql("CREATE SCHEMA IF NOT EXISTS gs_carbon.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS gs_carbon.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gs_carbon.gold")

In [0]:
spark.sql("USE CATALOG gs_carbon")
spark.sql("USE SCHEMA bronze")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS gs_carbon.bronze.rw;
SHOW VOLUMES IN gs_carbon.bronze;

In [0]:
display(dbutils.fs.ls("/Volumes/gs_carbon/bronze/rw"))

In [0]:
from pyspark.sql.functions import current_timestamp

df = (
    spark.read
         .format("csv")
         .option("header", "true")
         .option("inferSchema", "false")
         .load("dbfs:/Volumes/gs_carbon/bronze/rw/allprojects.csv")
)

df = df.toDF(
    *[
        c.strip()
         .replace(" ", "_")
         .replace("-", "_")
         .replace("(", "")
         .replace(")", "")
         .replace("/", "_")
        for c in df.columns
    ]
)

df = df.withColumn("dt_ingestion", current_timestamp())

(
    df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable("gs_carbon.bronze.verra_projects")
)

display(df)